# Qwen3-ASR on Amazon SageMaker with Bidirectional Streaming

Deploy Qwen3-ASR as a SageMaker real-time endpoint with [bidirectional streaming](https://aws.amazon.com/blogs/machine-learning/introducing-bidirectional-streaming-for-real-time-inference-on-amazon-sagemaker-ai/) for low-latency, real-time speech-to-text. Audio is streamed to the endpoint in chunks and partial transcriptions are returned as each chunk is processed, enabling live captioning and voice assistant use cases.

## Architecture

```
                                                   ┌────────────────────────────────────────┐
                                                   │  SageMaker Container (port 8080)       │
 Client (HTTP/2)  ──►  SageMaker Router  ──►  WS  │                                        │
                        (port 8443)                 │  GET  /ping                   → 200 OK │
                                                   │  WS   /invocations-bidirectional-stream │
                                                   │         ├─ Receives: JSON + PCM audio   │
                                                   │         └─ Sends:    JSON transcriptions│
                                                   └────────────────────────────────────────┘
```

Clients connect to SageMaker's HTTP/2 endpoint on port 8443 using the experimental `aws-sdk-sagemaker-runtime-http2` SDK. SageMaker routes traffic to a WebSocket server running inside the container on port 8080. The container uses FastAPI + uvicorn for native async WebSocket support, and runs Qwen3-ASR inference via the vLLM backend.

## Prerequisites

- **AWS CLI** configured (`aws configure`)
- **Docker** installed (for local builds) or **AWS CodeBuild** (for remote builds)
- **Python 3.9+**
- **IAM permissions**: `AmazonSageMakerFullAccess`, `sagemaker:InvokeEndpointWithResponseStream`, and ECR push permissions
- **S3 bucket** for the model artifact
- **GPU instance quota**: `ml.g6.xlarge` (or larger) in your target region

## File Overview

| File | Description |
|------|-------------|
| `app.py` | FastAPI WebSocket server bridging SageMaker bidirectional streaming to Qwen3-ASR |
| `Dockerfile` | Container image based on NVIDIA CUDA 12.8 with vLLM, CUDA compat package, and bidirectional streaming label |
| `entrypoint.sh` | Container entrypoint that enables CUDA forward compatibility before starting the app |
| `build_and_push.sh` | Shell script to build the Docker image locally and push to ECR |
| `buildspec.yml` | AWS CodeBuild spec for building and pushing the container remotely |
| `deploy.py` | Python script to create the SageMaker model, endpoint config, and endpoint |
| `client.py` | Example Python client for invoking the endpoint with real-time audio streaming |
| `test_local.py` | Integration tests for the WebSocket protocol (runs against mock mode, no GPU needed) |

# Step 1: Prepare the Model Artifact

SageMaker downloads model.tar.gz from S3 and extracts it to /opt/ml/model/ in the container before starting the application.

In [ ]:
# Install the Hugging Face CLI
# per https://github.com/huggingface/huggingface_hub/issues/3484#issuecomment-3452532430
# The recommended way to use curl and we should use hf instead of huggingface-cli command
!curl -LsSf https://hf.co/cli/install.sh | sh

# Add ~/.local/bin to PATH for this notebook session so !hf works in subsequent cells
import os
from pathlib import Path
local_bin = str(Path.home() / ".local" / "bin")
if local_bin not in os.environ["PATH"]:
    os.environ["PATH"] = local_bin + os.pathsep + os.environ["PATH"]

In [ ]:
# Download model weights from HuggingFace
!hf download Qwen/Qwen3-ASR-1.7B --local-dir ./Qwen3-ASR-1.7B   

# Create the tarball (must be created from inside the model directory)
# This may take minutes depending on the notebook instance you are using
!cd Qwen3-ASR-1.7B && tar -czf ../model.tar.gz .

# Upload to S3, please replace the bucket name with your own
!aws s3 cp model.tar.gz s3://<your-bucket>/qwen3-asr/model.tar.gz

# Step 2: Build and Push the Container
The container image is ~14 GB due to the CUDA toolkit and vLLM dependencies. Choose one of the following build methods:


## Option A: AWS CodeBuild (Recommended for non-x86 machines)
Use buildspec.yml with CodeBuild for faster, native linux/amd64 builds. Set up a CodeBuild project with:

In [ ]:
# Zip and upload source to S3 
# This may take a while due to downloading the model weight and zipping the model 

import boto3, sagemaker
import os                                     

session = boto3.Session()
account_id = session.client('sts').get_caller_identity()['Account']          
region = session.region_name                                  
# Update with your bucket name, it is the same bucket name in Step 1                                  
bucket = '<your-bucket>'           
print(f"Account: {account_id}, Region: {region}")  


# zip everything except the model weight and the model.tar.gz
!zip -r sagemaker-build.zip . -x "./Qwen3-ASR-1.7B/*" -x "./model.tar.gz"

!aws s3 cp ./sagemaker-build.zip s3://{bucket}/qwen3-asr/sagemaker-build.zip 

In [ ]:
# Create ECR repo

ecr = session.client('ecr')
try:
  ecr.create_repository(repositoryName='qwen3-asr-sagemaker')
  print("ECR repo created")

except ecr.exceptions.RepositoryAlreadyExistsException:
  print("ECR repo already exists")

In [ ]:
# Create CodeBuild project (one-time)

cb = session.client('codebuild')

# Use the notebook instance's role, or specify your own
# You can find it in: Notebook Instance > Permissions > IAM role ARN
notebook_role_arn = sagemaker.get_execution_role()

# Please make sure you have sufficient policy permission for sagemaker execution role and trust policy setup so that codebuild can assume this role as well

try:
  cb.create_project(
      name='sagemaker-qwen3-asr-container-build',
      source={
          'type': 'S3',
          'location': f'{bucket}/qwen3-asr/sagemaker-build.zip',
          'buildspec': 'buildspec.yml'
      },
      environment={
          'type': 'LINUX_CONTAINER',
          'image': 'aws/codebuild/standard:7.0',
          'computeType': 'BUILD_GENERAL1_LARGE',
          'privilegedMode': True,  # Required for Docker
      },
      serviceRole=notebook_role_arn,
      artifacts={'type': 'NO_ARTIFACTS'},
      timeoutInMinutes=60,
  )
  print("CodeBuild project created")
except cb.exceptions.ResourceAlreadyExistsException:
  print("CodeBuild project already exists")


In [ ]:
# Please register a free Docker Hub account and create a token so you are not get rate limit from docker hub
# pass your docker hub username and token as part of build environment variables
# Please check the codebuiler console to monitor the progress
response = cb.start_build(
    projectName='sagemaker-qwen3-asr-container-build',
    environmentVariablesOverride=[
        {
            'name': 'DOCKERHUB_USERNAME',
            'value': 'xxx',
            'type': 'PLAINTEXT'
        },
        {
            'name': 'DOCKERHUB_TOKEN',
            'value': 'xxxx',
            'type': 'PLAINTEXT'
        },
    ]
)
build_id = response['build']['id']
print(f"Build started: {build_id}")

## Option B: Local Docker Build
Requires Docker with linux/amd64 build support. On Apple Silicon Macs, this uses emulation and may be slow and can take hour

In [ ]:
!./build_and_push.sh

The script creates the ECR repository if needed, authenticates Docker, builds the image, and pushes. Note the image URI printed at the end.

## Step 3: Deploy the Endpoint

In [ ]:
#this will deploy a real-time sagemaker endpoint, in this case we are using a  ml.g6.xlarge sagemaker AI instance
!python deploy.py \
    --image-uri <IMAGE_URI_FROM_STEP_2> \
    --model-data-url s3://<your-bucket>/qwen3-asr/model.tar.gz \
    --role arn:aws:iam::<ACCOUNT_ID>:role/<SageMakerExecutionRole> \
    --instance-type ml.g6.xlarge \
    --endpoint-name qwen3-asr-bidi-streaming \
    --region us-east-1 \
    --wait

### Instance Type Selection

| Instance | GPU | VRAM | NVIDIA Driver | CUDA | Status |
|----------|-----|------|---------------|------|--------|
| `ml.g6.xlarge` | L4 | 24 GB | 535 | 12.2 | **Recommended** |
| `ml.g6.2xlarge` | L4 | 24 GB | 535 | 12.2 | More CPU/RAM |
| `ml.p5.xlarge` | H100 | 80 GB | 535 | 12.2 | High throughput |
| `ml.g5.xlarge` | A10G | 24 GB | 470 | 11.4 | **Not compatible** |

**Important**: `ml.g5.*` instances ship with NVIDIA driver 470 (CUDA 11.4), which is too old for vLLM. The CUDA forward compatibility package cannot bridge the CUDA 11 → 12 major version gap. Use `ml.g6.*` or newer instances instead.

The container includes the [CUDA forward compatibility package](https://docs.nvidia.com/deploy/cuda-compatibility/index.html) (`cuda-compat-12-8`) and an entrypoint script that automatically enables it when the host driver is older than the container's CUDA 12.8 toolkit. This bridges the gap between the host's CUDA 12.2 (driver 535 on g6) and the container's CUDA 12.8. See the [SageMaker GPU driver documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/inference-gpu-drivers.html) for driver versions by instance type.

### Startup Timing

Endpoint creation involves provisioning the instance, pulling the container image (~14 GB), downloading and extracting the model artifact (~3.5 GB), loading the model into GPU memory, and compiling CUDA graphs. Expect the endpoint to take 8-15 minutes to reach `InService`.

In [ ]:
## Step 4: Invoke the Endpoint
#Install the client dependencies:

!pip install aws-sdk-sagemaker-runtime-http2 soundfile scipy boto3

In [ ]:
!sudo yum install -y libsndfile

In [ ]:
# test the speach to text endpoint
!python client.py --endpoint-name <endpoint-name> --audio-file ./test.wav --region us-east-1

2026-03-31 10:18:05,253 [INFO] Found credentials from IAM Role: BaseNotebookInstanceEc2InstanceRole
2026-03-31 10:18:05,333 [INFO] Authenticated as: arn:aws:sts::669707688928:assumed-role/AmazonSageMaker-ExecutionRole-20260330T181586/SageMaker
2026-03-31 10:18:05,335 [INFO] Loading audio: ./test.wav
2026-03-31 10:18:07,028 [INFO] Audio loaded: 33.62 seconds (537970 samples)
2026-03-31 10:18:07,035 [INFO] Connecting to endpoint: qwen3-asr-bidi-streaming
2026-03-31 10:18:07,630 [INFO] Session started
2026-03-31 10:18:07,631 [INFO] Sent chunk 1 (32000 bytes)
2026-03-31 10:18:07,684 [INFO] [partial] lang= text=
2026-03-31 10:18:08,133 [INFO] Sent chunk 2 (32000 bytes)
2026-03-31 10:18:08,141 [INFO] [partial] lang= text=
2026-03-31 10:18:08,635 [INFO] Sent chunk 3 (32000 bytes)
2026-03-31 10:18:08,643 [INFO] [partial] lang= text=
2026-03-31 10:18:09,137 [INFO] Sent chunk 4 (32000 bytes)
2026-03-31 10:18:09,639 [INFO] Sent chunk 5 (32000 bytes)
2026-03-31 10:18:10,141 [INFO] Sent chunk 6 (32